# Kharita : parcours en six tâches

Les valeurs manquantes sont conservées, jamais transformées automatiquement en zéro. Les modèles utilisent une séparation des communes et les caractéristiques antérieures à la cible.

## Évaluation et interprétation

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"Data/resultats-elections-communales-maroc-2015-2021.csv").exists())
sys.path.insert(0,str(ROOT/"Script"))
from kharita_pipeline import load_raw,clean_data,make_comparison,build_features,save_clean_outputs
import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns
sns.set_theme(style="whitegrid")
raw=load_raw(); df,quality,party_cols=clean_data(raw)
print(f"{len(df):,} lignes; années {quality['years']}")

## Métriques, importances et clusters

MAE/RMSE/R² évaluent la régression. L’importance par permutation reflète la dépendance du modèle, pas une causalité.

## Évaluation et interprétation

Le notebook suit les choix de nettoyage documentés; les champs absents restent manquants et les analyses restent descriptives.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.inspection import permutation_importance
features=build_features(df,party_cols); data=features.dropna(subset=["turnout_2021"]); y=data.turnout_2021; X=data.drop(columns="turnout_2021"); cat=["region"]; num=[c for c in X if c not in cat]
pre=ColumnTransformer([("num",make_pipeline(SimpleImputer(strategy="median"),StandardScaler()),num),("cat",make_pipeline(SimpleImputer(strategy="most_frequent"),OneHotEncoder(handle_unknown="ignore")),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42); model=make_pipeline(pre,RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=42,n_jobs=-1)); model.fit(Xtr,ytr); pred=model.predict(Xte)
print("MAE",mean_absolute_error(yte,pred),"RMSE",mean_squared_error(yte,pred)**.5,"R2",r2_score(yte,pred))
plt.scatter(yte,pred,alpha=.5); plt.plot([0,100],[0,100],"--",color="black"); plt.xlabel("Participation réelle 2021 (%)"); plt.ylabel("Prédiction (%)"); plt.show()
perm=permutation_importance(model,Xte,yte,n_repeats=10,random_state=42,scoring="neg_mean_absolute_error",n_jobs=-1); display(pd.Series(perm.importances_mean,index=X.columns).sort_values(ascending=False).head(15).to_frame("importance_permutation"))
from sklearn.cluster import KMeans
w=make_comparison(df); cols=["turnout_pct_2015","turnout_pct_2021","variation_points","party_concentration_2015","winning_margin_2015"]; cl=w[cols].dropna(); Z=StandardScaler().fit_transform(cl); labels=KMeans(n_clusters=3,n_init=20,random_state=42).fit_predict(Z); cl=cl.assign(cluster=labels); display(cl.groupby("cluster").mean(numeric_only=True).round(3))
print("Interprétation limitée à une expérience 2015→2021; aucune causalité ni prévision générale.")